# R1 — Paper-Style Absolute Target Ablation

Single-variable experiment: only the prediction target changes. Universe, feature set, walk-forward splits, model configs, seeds, and bootstrap methodology are all copied verbatim from `feature_diagnostics.ipynb` / `neural_network_model.ipynb`. See `docs/weekly_sector_rotation_r1_target_test.md` for the full write-up; this notebook is the executable source of the CSVs it cites.

In [1]:
"""
R1: paper-style absolute +1% next-week target vs. existing VTI-relative target.
Single-variable experiment -- target only. All model/feature/split code copied
verbatim from feature_diagnostics.ipynb / neural_network_model.ipynb.
"""
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()

REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

SECTOR_ETFS = ["VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE"]
BENCHMARK_ETF = "VTI"
SYMBOLS = SECTOR_ETFS + [BENCHMARK_ETF]

# ---------------------------------------------------------------------------
# 1. Load panel (unchanged, existing validated data) + raw daily OHLC (for the
#    paper-style open/close execution target only -- features are untouched).
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

AUDIT_COLUMNS = [
    "week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
    "next_week_first_trading_date", "next_week_last_trading_date",
]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

raw_frames = []
for s in SYMBOLS:
    df = pd.read_csv(PRICES_ROOT / f"{s}.csv", parse_dates=["date"])
    df["symbol"] = s
    raw_frames.append(df[["symbol", "date", "open", "close"]])
raw_long = pd.concat(raw_frames, ignore_index=True)

## 2. Build the three targets.

In [2]:
# 2a. Existing target (unchanged formula, renamed for clarity in this comparison).
panel["next_week_excess_return"] = panel.groupby("symbol")["excess_return_vs_vti_1w"].shift(-1)
panel["target_outperform_vti"] = (panel["next_week_excess_return"] > 0).astype("Int64")
panel.loc[panel["next_week_excess_return"].isna(), "target_outperform_vti"] = pd.NA

# 2b. Primary paper-style target: next week's first-trading-day open -> final-trading-day close,
#     using RAW (unadjusted-for-dividends, split-adjusted) open/close -- the actual price levels a
#     trader entering at next Monday's open and exiting at next Friday's close would realize. Adjusted
#     close is deliberately NOT used here: it back-adjusts for future dividends, which is not what an
#     open->close execution return over a single week actually pays/receives.
panel = panel.merge(
    raw_long.rename(columns={"date": "next_week_first_trading_date", "open": "next_week_first_open"})
    [["symbol", "next_week_first_trading_date", "next_week_first_open"]],
    on=["symbol", "next_week_first_trading_date"], how="left",
)
panel = panel.merge(
    raw_long.rename(columns={"date": "next_week_last_trading_date", "close": "next_week_final_close"})
    [["symbol", "next_week_last_trading_date", "next_week_final_close"]],
    on=["symbol", "next_week_last_trading_date"], how="left",
)
panel["next_week_open_to_close_return"] = panel["next_week_final_close"] / panel["next_week_first_open"] - 1
panel["target_abs_1pct_next_week"] = (panel["next_week_open_to_close_return"] >= 0.01).astype("Int64")
panel.loc[panel["next_week_open_to_close_return"].isna(), "target_abs_1pct_next_week"] = pd.NA

# 2c. Friday-to-Friday +1% sensitivity target, using the panel's own weekly adjusted-close series
#     (abs_return_1w is already sector-own weekly adjusted-close return; shift -1 for next week's).
panel["next_week_fri_to_fri_return"] = panel.groupby("symbol")["abs_return_1w"].shift(-1)
panel["target_abs_1pct_fri_to_fri"] = (panel["next_week_fri_to_fri_return"] >= 0.01).astype("Int64")
panel.loc[panel["next_week_fri_to_fri_return"].isna(), "target_abs_1pct_fri_to_fri"] = pd.NA

# Derived (algebra on existing leak-free columns, no new data pull): next week's own VTI return,
# used later for "excess return vs VTI" on the Top-K economic metrics.
panel["next_week_vti_return"] = panel["next_week_fri_to_fri_return"] - panel["next_week_excess_return"]

# No target may ever be used as a feature.
assert not (set(["next_week_excess_return", "target_outperform_vti", "next_week_open_to_close_return",
                  "target_abs_1pct_next_week", "next_week_fri_to_fri_return", "target_abs_1pct_fri_to_fri",
                  "next_week_vti_return", "next_week_first_open", "next_week_final_close"]) & set(FEATURE_COLUMNS))

# Lookahead check for the primary target: its own execution period must start strictly after this
# row's feature cutoff -- restate the panel's own guarantee for the new target explicitly.
chk = panel.dropna(subset=["next_week_open_to_close_return", "next_week_first_trading_date"])
assert (chk["feature_cutoff_timestamp"] < chk["next_week_first_trading_date"]).all(), "lookahead in primary target"

print(f"[{time.time()-t0:.0f}s] targets built. panel shape={panel.shape}")

[0s] targets built. panel shape=(12551, 53)


## 3. STEP 4 -- target quality audit.

In [3]:
model_data_all = panel.dropna(subset=FEATURE_COLUMNS).copy()
model_data_all["year"] = model_data_all["date"].dt.year

MIN_TRAIN_YEARS = 3
years = sorted(model_data_all["year"].unique())
test_years = years[MIN_TRAIN_YEARS:]
model_data_all["fold"] = np.where(model_data_all["year"].isin(test_years), model_data_all["year"], pd.NA)

TARGET_COLS = {
    "existing_vti_relative": "target_outperform_vti",
    "primary_abs_1pct_open_close": "target_abs_1pct_next_week",
    "sensitivity_abs_1pct_fri_fri": "target_abs_1pct_fri_to_fri",
}

quality_rows = []
for name, col in TARGET_COLS.items():
    s = panel[col]
    n_total = len(panel)
    n_missing = s.isna().sum()
    n_valid = n_total - n_missing
    n_pos = (s == 1).sum()
    n_neg = (s == 0).sum()
    quality_rows.append({
        "target": name, "column": col, "n_total_rows": n_total, "n_valid_labels": n_valid,
        "n_missing_labels": n_missing, "n_positive": n_pos, "n_negative": n_neg,
        "positive_pct": n_pos / n_valid if n_valid else np.nan,
    })
target_quality_summary = pd.DataFrame(quality_rows)
target_quality_summary.to_csv(OUTPUTS_ROOT / "r1_target_quality_summary.csv", index=False)

balance_rows = []
for name, col in TARGET_COLS.items():
    valid = panel.dropna(subset=[col])
    by_etf = valid.groupby("symbol")[col].agg(["count", "mean"]).rename(columns={"mean": "positive_pct"})
    by_etf["target"] = name
    by_etf["dimension"] = "etf"
    by_etf = by_etf.reset_index().rename(columns={"symbol": "key"})

    valid_y = valid.copy()
    valid_y["year"] = valid_y["date"].dt.year
    by_year = valid_y.groupby("year")[col].agg(["count", "mean"]).rename(columns={"mean": "positive_pct"})
    by_year["target"] = name
    by_year["dimension"] = "year"
    by_year = by_year.reset_index().rename(columns={"year": "key"})

    valid_f = valid.merge(model_data_all[["date", "symbol", "fold"]], on=["date", "symbol"], how="inner")
    valid_f = valid_f.dropna(subset=["fold"])
    by_fold = valid_f.groupby("fold")[col].agg(["count", "mean"]).rename(columns={"mean": "positive_pct"})
    by_fold["target"] = name
    by_fold["dimension"] = "walk_forward_test_fold"
    by_fold = by_fold.reset_index().rename(columns={"fold": "key"})

    balance_rows.extend([by_etf, by_year, by_fold])
class_balance_detail = pd.concat(balance_rows, ignore_index=True)
class_balance_detail.to_csv(OUTPUTS_ROOT / "r1_class_balance_detail.csv", index=False)

# Disagreement matrices.
def disagreement_matrix(a_col, b_col, a_name, b_name):
    both = panel.dropna(subset=[a_col, b_col])
    a = both[a_col].astype(int)
    b = both[b_col].astype(int)
    n = len(both)
    both_pos = int(((a == 1) & (b == 1)).sum())
    both_neg = int(((a == 0) & (b == 0)).sum())
    a_pos_b_neg = int(((a == 1) & (b == 0)).sum())
    a_neg_b_pos = int(((a == 0) & (b == 1)).sum())
    disagree = a_pos_b_neg + a_neg_b_pos
    return {
        "comparison": f"{a_name}_vs_{b_name}", "n_rows": n,
        "both_positive": both_pos, "both_negative": both_neg,
        f"{a_name}_positive_{b_name}_negative": a_pos_b_neg,
        f"{a_name}_negative_{b_name}_positive": a_neg_b_pos,
        "overall_disagreement_pct": disagree / n if n else np.nan,
    }

disagreement_A = disagreement_matrix("target_outperform_vti", "target_abs_1pct_next_week",
                                      "existing_relative", "primary_absolute")
disagreement_B = disagreement_matrix("target_abs_1pct_next_week", "target_abs_1pct_fri_to_fri",
                                      "primary_absolute", "fri_fri_sensitivity")
disagreement_summary = pd.DataFrame([disagreement_A, disagreement_B])
disagreement_summary.to_csv(OUTPUTS_ROOT / "r1_target_disagreement_summary.csv", index=False)

print(f"[{time.time()-t0:.0f}s] Step 4 (target quality audit) done.")

[0s] Step 4 (target quality audit) done.


## 4. STEP 5 -- return-distribution audit for the primary target's underlying return.

In [4]:
ret = panel.dropna(subset=["next_week_open_to_close_return"])
pcts = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

def dist_row(s, label):
    q = s.quantile(pcts)
    row = {"group": label, "n": len(s), "mean": s.mean(), "median": s.median(), "std": s.std()}
    row.update({f"p{int(p*100)}": q.loc[p] for p in pcts})
    return row

dist_rows = [dist_row(ret["next_week_open_to_close_return"], "ALL")]
for etf, g in ret.groupby("symbol"):
    dist_rows.append(dist_row(g["next_week_open_to_close_return"], etf))
return_distribution_summary = pd.DataFrame(dist_rows)
return_distribution_summary.to_csv(OUTPUTS_ROOT / "r1_primary_return_distribution.csv", index=False)

n_missing_primary_only = int((panel["next_week_last_trading_date"].notna() &
                               panel["next_week_final_close"].isna()).sum())
print(f"[{time.time()-t0:.0f}s] Step 5 (return distribution) done. "
      f"data-quality gaps beyond the final-week edge: {n_missing_primary_only} row(s)")

[0s] Step 5 (return distribution) done. data-quality gaps beyond the final-week edge: 1 row(s)


## 5. Feature groups / cumulative feature-set steps (verbatim from feature_diagnostics.ipynb).

In [5]:
FEATURE_GROUPS = {
    "A": ["abs_return_1w", "abs_return_2w", "abs_return_4w", "abs_return_8w", "abs_return_12w", "abs_return_26w"],
    "B": ["excess_return_vs_vti_1w", "excess_return_vs_vti_2w", "excess_return_vs_vti_4w",
          "excess_return_vs_vti_8w", "excess_return_vs_vti_12w", "excess_return_vs_vti_26w"],
    "C": ["cross_sectional_rank_1w", "cross_sectional_rank_4w", "cross_sectional_rank_8w", "cross_sectional_rank_12w"],
    "D": ["realized_vol_4w", "realized_vol_12w", "realized_vol_26w"],
    "E": ["trend_dist_from_ma_4w", "trend_dist_from_ma_12w", "trend_dist_from_ma_26w"],
    "F": ["drawdown_dist_from_high_13w", "drawdown_dist_from_high_26w"],
    "G": ["volume_change_1w", "volume_vs_avg_4w", "volume_vs_avg_12w"],
    "H": ["vti_return_1w", "vti_return_4w", "vti_return_12w", "vti_volatility_4w"],
    "I": ["sector_return_dispersion_1w", "average_sector_return_1w", "sectors_outperforming_vti_1w"],
}
FEATURE_SET_STEPS = []
_added = []
for g in ["B", "A", "C", "D", "E", "F", "G", "H", "I"]:
    _added.append(g)
    ordered = sorted(_added, key=lambda x: "ABCDEFGHI".index(x))
    label = "+".join(ordered)
    cols = [c for gg in ordered for c in FEATURE_GROUPS[gg]]
    FEATURE_SET_STEPS.append((label, cols))

RANK_K_VALUES = [2, 3]

# ---------------------------------------------------------------------------
# 6. Walk-forward evaluators (verbatim logic from the existing notebooks; only the
#    label column and target name are parameterized -- nothing else changes).
# ---------------------------------------------------------------------------

def build_model_data(label_col: str) -> pd.DataFrame:
    md = panel.dropna(subset=FEATURE_COLUMNS + [label_col]).copy()
    md["label_binary"] = md[label_col].astype(int)
    md["year"] = md["date"].dt.year
    return md


def top_bottom_spread_from_scores(frame: pd.DataFrame, score_col: str, k: int) -> pd.Series:
    ranked = frame[score_col].rank(ascending=False, method="first")
    n = len(frame)
    top_return = frame.loc[ranked <= k, "next_week_excess_return_for_eval"].mean()
    bottom_return = frame.loc[ranked > n - k, "next_week_excess_return_for_eval"].mean()
    return pd.Series({"top_return": top_return, "bottom_return": bottom_return})


def walk_forward_evaluate_lr(md: pd.DataFrame, years_, test_years_, feature_cols):
    oof_frames = []
    fold_rows = []
    for test_year in test_years_:
        train_mask = md["year"] < test_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_train = scaler.transform(md.loc[train_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_train = md.loc[train_mask, "label_binary"]
        clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
        predicted_proba = clf.predict_proba(X_test)[:, 1]

        fold = md.loc[test_mask, ["date", "symbol", "next_week_excess_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

        y_test = md.loc[test_mask, "label_binary"]
        fold_auc = roc_auc_score(y_test, predicted_proba) if y_test.nunique() > 1 else np.nan
        fold_rows.append({
            "test_year": test_year, "train_start_year": int(years_[0]), "train_end_year": test_year - 1,
            "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum()),
            "train_positive_rate": float(y_train.mean()), "test_positive_rate": float(y_test.mean()),
            "fold_auc": fold_auc,
        })

    oof = pd.concat(oof_frames, ignore_index=True)
    oof["next_week_excess_return_for_eval"] = oof["next_week_excess_return"]
    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()
    metrics = {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), **spread_stats}
    return metrics, oof, pd.DataFrame(fold_rows)


NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features, seed=NN_SEED):
    torch.manual_seed(seed)
    model = SmallMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    for _ in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


def walk_forward_evaluate_nn(md, years_, test_years_, feature_cols):
    oof_frames = []
    fold_rows = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()
        model = train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features=len(feature_cols))
        with torch.no_grad():
            predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()
        fold = md.loc[test_mask, ["date", "symbol", "next_week_excess_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

        y_test = md.loc[test_mask, "label_binary"]
        fold_auc = roc_auc_score(y_test, predicted_proba) if y_test.nunique() > 1 else np.nan
        fold_rows.append({
            "test_year": test_year, "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum()),
            "train_positive_rate": float(md.loc[fit_mask, "label_binary"].mean()),
            "test_positive_rate": float(y_test.mean()), "fold_auc": fold_auc,
        })

    oof = pd.concat(oof_frames, ignore_index=True)
    oof["next_week_excess_return_for_eval"] = oof["next_week_excess_return"]
    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()
    metrics = {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), **spread_stats}
    return metrics, oof, pd.DataFrame(fold_rows)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, seed):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    for _ in range(V2_MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= V2_PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


def walk_forward_evaluate_nn_v2(md, years_, test_years_, feature_cols):
    oof_frames = []
    fold_rows = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()
        seed_predictions = []
        for seed_offset in range(N_SEEDS):
            model = train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset)
            with torch.no_grad():
                seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
        predicted_proba = np.mean(seed_predictions, axis=0)
        fold = md.loc[test_mask, ["date", "symbol", "next_week_excess_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

        y_test = md.loc[test_mask, "label_binary"]
        fold_auc = roc_auc_score(y_test, predicted_proba) if y_test.nunique() > 1 else np.nan
        fold_rows.append({
            "test_year": test_year, "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum()),
            "train_positive_rate": float(md.loc[fit_mask, "label_binary"].mean()),
            "test_positive_rate": float(y_test.mean()), "fold_auc": fold_auc,
        })

    oof = pd.concat(oof_frames, ignore_index=True)
    oof["next_week_excess_return_for_eval"] = oof["next_week_excess_return"]
    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()
    metrics = {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), **spread_stats}
    return metrics, oof, pd.DataFrame(fold_rows)


BLOCK_SIZE = 8
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260812


def block_bootstrap_ci(values: pd.Series, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                        ci: float = 0.95, seed: int = BOOTSTRAP_SEED) -> dict:
    arr = values.to_numpy()
    n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(np.ceil(n / block_size))
    max_start = n - block_size
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s: s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()
    lower, upper = np.percentile(boot_means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return {
        "point_estimate": arr.mean(), "ci_lower": lower, "ci_upper": upper,
        "n_weeks": n, "significant": bool(lower > 0 or upper < 0),
    }

## 7. Run: primary target, full 9-step ablation, 3 models (LR, NN v1, NN v2).

In [6]:
def run_target(label_col: str, target_name: str, run_all_feature_steps: bool):
    md = build_model_data(label_col)
    yrs = sorted(md["year"].unique())
    ty = yrs[MIN_TRAIN_YEARS:]
    steps = FEATURE_SET_STEPS if run_all_feature_steps else FEATURE_SET_STEPS[-1:]

    lr_rows, nn1_rows, nn2_rows = [], [], []
    lr_folds_all, nn1_folds_all, nn2_folds_all = [], [], []
    oof_by_model = {}
    for label, cols in steps:
        m, oof, folds = walk_forward_evaluate_lr(md, yrs, ty, cols)
        lr_rows.append({"target": target_name, "feature_set": label, "n_features": len(cols), **m})
        folds.insert(0, "feature_set", label); folds.insert(0, "target", target_name)
        lr_folds_all.append(folds)
        if label == steps[-1][0]:
            oof_by_model[("LR", label)] = oof
        print(f"  [{time.time()-t0:.0f}s] {target_name} LR {label}: auc={m['auc']:.4f} k3={m['mean_spread_k3']:.5f}")

        m, oof, folds = walk_forward_evaluate_nn(md, yrs, ty, cols)
        nn1_rows.append({"target": target_name, "feature_set": label, "n_features": len(cols), **m})
        folds.insert(0, "feature_set", label); folds.insert(0, "target", target_name)
        nn1_folds_all.append(folds)
        if label == steps[-1][0]:
            oof_by_model[("NNv1", label)] = oof
        print(f"  [{time.time()-t0:.0f}s] {target_name} NNv1 {label}: auc={m['auc']:.4f} k3={m['mean_spread_k3']:.5f}")

        m, oof, folds = walk_forward_evaluate_nn_v2(md, yrs, ty, cols)
        nn2_rows.append({"target": target_name, "feature_set": label, "n_features": len(cols), **m})
        folds.insert(0, "feature_set", label); folds.insert(0, "target", target_name)
        nn2_folds_all.append(folds)
        if label == steps[-1][0]:
            oof_by_model[("NNv2", label)] = oof
        print(f"  [{time.time()-t0:.0f}s] {target_name} NNv2 {label}: auc={m['auc']:.4f} k3={m['mean_spread_k3']:.5f}")

    return {
        "lr": pd.DataFrame(lr_rows), "nn1": pd.DataFrame(nn1_rows), "nn2": pd.DataFrame(nn2_rows),
        "lr_folds": pd.concat(lr_folds_all, ignore_index=True),
        "nn1_folds": pd.concat(nn1_folds_all, ignore_index=True),
        "nn2_folds": pd.concat(nn2_folds_all, ignore_index=True),
        "oof": oof_by_model, "model_data": md, "full_feature_label": steps[-1][0],
    }


print(f"[{time.time()-t0:.0f}s] Running PRIMARY target (full 9-step ablation)...")
primary_results = run_target("target_abs_1pct_next_week", "primary_abs_1pct_open_close", run_all_feature_steps=True)

print(f"[{time.time()-t0:.0f}s] Running SENSITIVITY target (full-feature-set only)...")
sensitivity_results = run_target("target_abs_1pct_fri_to_fri", "sensitivity_abs_1pct_fri_fri", run_all_feature_steps=False)

for tag, res in [("primary", primary_results), ("sensitivity", sensitivity_results)]:
    res["lr"].to_csv(OUTPUTS_ROOT / f"r1_{tag}_lr_ablation_walk_forward_results.csv", index=False)
    res["nn1"].to_csv(OUTPUTS_ROOT / f"r1_{tag}_nn_v1_ablation_walk_forward_results.csv", index=False)
    res["nn2"].to_csv(OUTPUTS_ROOT / f"r1_{tag}_nn_v2_ablation_walk_forward_results.csv", index=False)
    per_fold = pd.concat([
        res["lr_folds"].assign(model="LR"),
        res["nn1_folds"].assign(model="NNv1"),
        res["nn2_folds"].assign(model="NNv2"),
    ], ignore_index=True)
    per_fold.to_csv(OUTPUTS_ROOT / f"r1_{tag}_per_fold_metrics.csv", index=False)

print(f"[{time.time()-t0:.0f}s] Step 6/7 (model comparison + walk-forward) done.")

[0s] Running PRIMARY target (full 9-step ablation)...


  [1s] primary_abs_1pct_open_close LR B: auc=0.5108 k3=0.00017


  [9s] primary_abs_1pct_open_close NNv1 B: auc=0.4995 k3=0.00050


  [41s] primary_abs_1pct_open_close NNv2 B: auc=0.5003 k3=0.00086


  [41s] primary_abs_1pct_open_close LR A+B: auc=0.5505 k3=0.00051


...

  [249s] primary_abs_1pct_open_close NNv1 A+B+C+D+E+F+G+H+I: auc=0.5694 k3=0.00049


  [264s] primary_abs_1pct_open_close NNv2 A+B+C+D+E+F+G+H+I: auc=0.5708 k3=0.00114
[264s] Running SENSITIVITY target (full-feature-set only)...


  [264s] sensitivity_abs_1pct_fri_fri LR A+B+C+D+E+F+G+H+I: auc=0.5338 k3=0.00064


  [269s] sensitivity_abs_1pct_fri_fri NNv1 A+B+C+D+E+F+G+H+I: auc=0.5673 k3=0.00055


  [283s] sensitivity_abs_1pct_fri_fri NNv2 A+B+C+D+E+F+G+H+I: auc=0.5672 k3=0.00140
[283s] Step 6/7 (model comparison + walk-forward) done.


## 8. STEP 8 -- Top-K evaluation, full-feature-set OOF predictions, both targets, all 3 models.

In [7]:
def topk_metrics(oof: pd.DataFrame, md: pd.DataFrame, k: int, target_name: str, model_name: str) -> dict:
    joined = oof.merge(
        md[["date", "symbol", "next_week_open_to_close_return", "next_week_fri_to_fri_return",
            "next_week_excess_return", "next_week_vti_return"]],
        on=["date", "symbol"], how="left", suffixes=("", "_md"),
    )
    # Use the same return series the target itself is defined on, for "actual next-week ETF returns".
    ret_col = "next_week_open_to_close_return" if target_name.startswith("primary") else (
        "next_week_fri_to_fri_return" if target_name.startswith("sensitivity") else "next_week_excess_return")

    weekly_precision, weekly_any_pos, weekly_all_pos = [], [], []
    weekly_mean_ret, weekly_all_sector_mean, weekly_vti_ret, weekly_bottomk_mean = [], [], [], []
    for date, g in joined.groupby("date"):
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        n = len(g)
        top = g.loc[ranked <= k]
        bottom = g.loc[ranked > n - k]
        if top[ret_col].isna().any() or top["label_binary"].isna().any():
            continue
        weekly_precision.append(top["label_binary"].mean())
        weekly_any_pos.append(float((top["label_binary"] == 1).any()))
        weekly_all_pos.append(float((top["label_binary"] == 1).all()))
        weekly_mean_ret.append(top[ret_col].mean())
        weekly_all_sector_mean.append(g[ret_col].mean())
        weekly_vti_ret.append(g["next_week_vti_return"].iloc[0])
        weekly_bottomk_mean.append(bottom[ret_col].mean())

    weekly_precision = pd.Series(weekly_precision)
    weekly_mean_ret = pd.Series(weekly_mean_ret)
    weekly_all_sector_mean = pd.Series(weekly_all_sector_mean)
    weekly_vti_ret = pd.Series(weekly_vti_ret)
    weekly_bottomk_mean = pd.Series(weekly_bottomk_mean)

    base_rate = md["label_binary"].mean()
    return {
        "target": target_name, "model": model_name, "k": k, "n_weeks": len(weekly_precision),
        "precision_at_k": weekly_precision.mean(), "base_positive_rate": base_rate,
        "precision_minus_base": weekly_precision.mean() - base_rate,
        "pct_weeks_at_least_one_positive": pd.Series(weekly_any_pos).mean(),
        "pct_weeks_all_positive": pd.Series(weekly_all_pos).mean(),
        "mean_selected_return": weekly_mean_ret.mean(), "median_selected_return": weekly_mean_ret.median(),
        "selected_minus_all_sector_mean": (weekly_mean_ret - weekly_all_sector_mean).mean(),
        "selected_minus_vti": (weekly_mean_ret - weekly_vti_ret).mean(),
        "topk_minus_bottomk_spread": (weekly_mean_ret - weekly_bottomk_mean).mean(),
        "win_rate_vs_vti": (weekly_mean_ret > weekly_vti_ret).mean(),
    }, {
        "precision_series": weekly_precision, "selected_ret_series": weekly_mean_ret,
        "all_sector_mean_series": weekly_all_sector_mean, "vti_ret_series": weekly_vti_ret,
        "bottomk_mean_series": weekly_bottomk_mean,
    }

topk_rows = []
bootstrap_rows = []
model_name_map = {"LR": "LR", "NNv1": "NN_v1", "NNv2": "NN_v2"}
for tag, res in [("primary_abs_1pct_open_close", primary_results), ("sensitivity_abs_1pct_fri_fri", sensitivity_results)]:
    full_label = res["full_feature_label"]
    for model_key in ["LR", "NNv1", "NNv2"]:
        oof = res["oof"][(model_key, full_label)]
        md = res["model_data"]
        for k in RANK_K_VALUES:
            row, series = topk_metrics(oof, md, k, tag, model_name_map[model_key])
            topk_rows.append(row)

            prec_boot = block_bootstrap_ci(series["precision_series"])
            base_rate = row["base_positive_rate"]
            bootstrap_rows.append({
                "target": tag, "model": model_name_map[model_key], "k": k, "quantity": "precision_minus_base",
                "point_estimate": prec_boot["point_estimate"] - base_rate,
                "ci_lower": prec_boot["ci_lower"] - base_rate, "ci_upper": prec_boot["ci_upper"] - base_rate,
                "n_weeks": prec_boot["n_weeks"], "significant": prec_boot["significant"],
            })
            edge_all = series["selected_ret_series"] - series["all_sector_mean_series"]
            b = block_bootstrap_ci(edge_all)
            bootstrap_rows.append({"target": tag, "model": model_name_map[model_key], "k": k,
                                    "quantity": "selected_minus_all_sector_mean", **b})
            edge_vti = series["selected_ret_series"] - series["vti_ret_series"]
            b = block_bootstrap_ci(edge_vti)
            bootstrap_rows.append({"target": tag, "model": model_name_map[model_key], "k": k,
                                    "quantity": "selected_minus_vti", **b})
            spread = series["selected_ret_series"] - series["bottomk_mean_series"]
            b = block_bootstrap_ci(spread)
            bootstrap_rows.append({"target": tag, "model": model_name_map[model_key], "k": k,
                                    "quantity": "topk_minus_bottomk_spread", **b})

topk_summary = pd.DataFrame(topk_rows)
topk_summary.to_csv(OUTPUTS_ROOT / "r1_topk_summary.csv", index=False)
r1_bootstrap_summary = pd.DataFrame(bootstrap_rows)
r1_bootstrap_summary.to_csv(OUTPUTS_ROOT / "r1_block_bootstrap_ci.csv", index=False)

print(f"[{time.time()-t0:.0f}s] Step 8/9 (Top-K + bootstrap) done.")

[296s] Step 8/9 (Top-K + bootstrap) done.


## 9. STEP 10 -- comparison table vs existing target (read existing baseline CSVs, unmodified).

In [8]:
existing_ablation = pd.read_csv(OUTPUTS_ROOT / "ablation_walk_forward_results.csv")
existing_nn1 = pd.read_csv(OUTPUTS_ROOT / "nn_ablation_walk_forward_results.csv")
existing_nn2 = pd.read_csv(OUTPUTS_ROOT / "nn_v2_ablation_walk_forward_results.csv")
existing_bootstrap_nn2 = pd.read_csv(OUTPUTS_ROOT / "nn_v2_block_bootstrap_ci.csv")

existing_pos_rate = model_data_all["target_outperform_vti"].mean() if "target_outperform_vti" in model_data_all else \
    panel["target_outperform_vti"].dropna().astype(int).mean()
primary_pos_rate = panel["target_abs_1pct_next_week"].dropna().astype(int).mean()

def best_row(df, metric="auc"):
    return df.loc[df[metric].idxmax()]

comp_rows = []
comp_rows.append({"metric": "positive_class_rate", "existing_vti_relative": float(existing_pos_rate),
                   "primary_abs_1pct": float(primary_pos_rate)})
comp_rows.append({"metric": "LR_AUC_best_of_9_sets", "existing_vti_relative": float(existing_ablation["auc"].max()),
                   "primary_abs_1pct": float(primary_results["lr"]["auc"].max())})
comp_rows.append({"metric": "NNv1_AUC_best_of_9_sets", "existing_vti_relative": float(existing_nn1["auc"].max()),
                   "primary_abs_1pct": float(primary_results["nn1"]["auc"].max())})
comp_rows.append({"metric": "NNv2_AUC_best_of_9_sets", "existing_vti_relative": float(existing_nn2["auc"].max()),
                   "primary_abs_1pct": float(primary_results["nn2"]["auc"].max())})
comp_rows.append({"metric": "best_AUC_any_model", "existing_vti_relative":
                   float(max(existing_ablation["auc"].max(), existing_nn1["auc"].max(), existing_nn2["auc"].max())),
                   "primary_abs_1pct": float(max(primary_results["lr"]["auc"].max(), primary_results["nn1"]["auc"].max(), primary_results["nn2"]["auc"].max()))})
comp_rows.append({"metric": "NNv1_AUC_fold_std_across_9_sets", "existing_vti_relative": float(existing_nn1["auc"].std()),
                   "primary_abs_1pct": float(primary_results["nn1"]["auc"].std())})
comp_rows.append({"metric": "NNv2_AUC_fold_std_across_9_sets", "existing_vti_relative": float(existing_nn2["auc"].std()),
                   "primary_abs_1pct": float(primary_results["nn2"]["auc"].std())})

for k in RANK_K_VALUES:
    ex_row = existing_bootstrap_nn2[existing_bootstrap_nn2["target"] == f"nn_v2_A+B+C+D+E+F+G+H+I_k{k}"]
    pr_row = r1_bootstrap_summary[(r1_bootstrap_summary["target"] == "primary_abs_1pct_open_close") &
                                   (r1_bootstrap_summary["model"] == "NN_v2") & (r1_bootstrap_summary["k"] == k) &
                                   (r1_bootstrap_summary["quantity"] == "topk_minus_bottomk_spread")]
    comp_rows.append({
        "metric": f"NNv2_bootstrap_topK_minus_bottomK_K{k}_point",
        "existing_vti_relative": float(ex_row["point_estimate"].iloc[0]) if len(ex_row) else np.nan,
        "primary_abs_1pct": float(pr_row["point_estimate"].iloc[0]) if len(pr_row) else np.nan,
    })
    comp_rows.append({
        "metric": f"NNv2_bootstrap_topK_minus_bottomK_K{k}_CI",
        "existing_vti_relative": f"[{ex_row['ci_lower'].iloc[0]:.6f}, {ex_row['ci_upper'].iloc[0]:.6f}]" if len(ex_row) else "n/a",
        "primary_abs_1pct": f"[{pr_row['ci_lower'].iloc[0]:.6f}, {pr_row['ci_upper'].iloc[0]:.6f}]" if len(pr_row) else "n/a",
    })

for k in RANK_K_VALUES:
    for model_key, label in [("LR", "LR"), ("NNv1", "NN_v1"), ("NNv2", "NN_v2")]:
        pr = topk_summary[(topk_summary["target"] == "primary_abs_1pct_open_close") &
                           (topk_summary["model"] == label) & (topk_summary["k"] == k)]
        if len(pr):
            comp_rows.append({"metric": f"{model_key}_precision@{k}_minus_base_primary_target",
                               "existing_vti_relative": "n/a (not computed for existing target in this pass)",
                               "primary_abs_1pct": float(pr["precision_minus_base"].iloc[0])})

r1_comparison_table = pd.DataFrame(comp_rows)
r1_comparison_table.to_csv(OUTPUTS_ROOT / "r1_comparison_vs_existing_target.csv", index=False)

## 10. STEP 12 -- primary vs sensitivity comparison table.

In [9]:
sens_comp_rows = []
sens_comp_rows.append({"metric": "positive_class_rate",
                        "primary": float(primary_pos_rate),
                        "sensitivity_fri_fri": float(panel["target_abs_1pct_fri_to_fri"].dropna().astype(int).mean())})
for model_key, label in [("lr", "LR"), ("nn1", "NNv1"), ("nn2", "NNv2")]:
    sens_comp_rows.append({
        "metric": f"{label}_AUC_full_feature_set",
        "primary": float(primary_results[model_key][primary_results[model_key]["feature_set"] == primary_results["full_feature_label"]]["auc"].iloc[0]),
        "sensitivity_fri_fri": float(sensitivity_results[model_key]["auc"].iloc[0]),
    })
r1_primary_vs_sensitivity = pd.DataFrame(sens_comp_rows)
r1_primary_vs_sensitivity.to_csv(OUTPUTS_ROOT / "r1_primary_vs_sensitivity_comparison.csv", index=False)

print(f"[{time.time()-t0:.0f}s] Step 10/12 (comparison tables) done.")

[296s] Step 10/12 (comparison tables) done.


## 11. Save the enriched panel slice (targets only, not full feature panel) for provenance.

In [10]:
target_cols_out = ["date", "symbol", "next_week_excess_return", "target_outperform_vti",
                    "next_week_first_open", "next_week_final_close", "next_week_open_to_close_return",
                    "target_abs_1pct_next_week", "next_week_fri_to_fri_return", "target_abs_1pct_fri_to_fri",
                    "next_week_vti_return"]
panel[target_cols_out].to_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", index=False)

summary = {
    "primary_positive_rate": float(primary_pos_rate),
    "existing_positive_rate": float(existing_pos_rate),
    "sensitivity_positive_rate": float(panel["target_abs_1pct_fri_to_fri"].dropna().astype(int).mean()),
    "primary_lr_best_auc": float(primary_results["lr"]["auc"].max()),
    "primary_nn1_best_auc": float(primary_results["nn1"]["auc"].max()),
    "primary_nn2_best_auc": float(primary_results["nn2"]["auc"].max()),
    "existing_lr_best_auc": float(existing_ablation["auc"].max()),
    "existing_nn1_best_auc": float(existing_nn1["auc"].max()),
    "existing_nn2_best_auc": float(existing_nn2["auc"].max()),
    "data_quality_gap_rows": n_missing_primary_only,
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r1_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2))

[296s] ALL DONE.
{
  "primary_positive_rate": 0.3497886593827259,
  "existing_positive_rate": 0.4952668516402807,
  "sensitivity_positive_rate": 0.3715311004784689,
  "primary_lr_best_auc": 0.5604441489617961,
  "primary_nn1_best_auc": 0.5711147853736089,
  "primary_nn2_best_auc": 0.5744853302500361,
  "existing_lr_best_auc": 0.5034370227870251,
  "existing_nn1_best_auc": 0.5149426346429351,
  "existing_nn2_best_auc": 0.5100043764435869,
  "data_quality_gap_rows": 1,
  "runtime_seconds": 295.9770839214325
}
